In [ ]:
import torch
import triton
import triton.language as tl

AUTOTUNE_MODE = "none"

In [ ]:
def get_fwd_autotune_configs():

    mode = AUTOTUNE_MODE

    if mode == "none":
        return [
            triton.Config(
                {"BLOCK_SIZE_Q": 64, "BLOCK_SIZE_KV": 32},
                num_stages = 2,
                num_warps = 4,
            )
        ]

    # minimal configs for "slow" mode (fewer configurations for faster tuning)
    if mode == "medium":
        return [
            triton.Config(
                {"BLOCK_SIZE_Q": BLOCK_SIZE_Q, "BLOCK_SIZE_KV": BLOCK_SIZE_KV},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE_Q in [32, 64]    # reduced set
            for BLOCK_SIZE_KV in [16, 32]   # reduced set
            for num_stages in [2, 3]        # reduced set
            if BLOCK_SIZE_KV < BLOCK_SIZE_Q
        ]

    # comprehensive configs for "max" mode (full search space)
    else:   # mode == "max" or any other value defaults to max
        return [
            triton.Config(
                {"BLOCK_SIZE_Q": BLOCK_SIZE_Q, "BLOCK_SIZE_KV": BLOCK_SIZE_KV},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE_Q in [16, 32, 64, 128]
            for BLOCK_SIZE_KV in [16, 32, 64]
            for num_stages in [2, 3, 4]
            for num_warps in [4, 8, 16]
            if BLOCK_SIZE_KV < BLOCK_SIZE_Q
        ]


In [ ]:
def get_preprocess_autotune_configs():

    mode = AUTOTUNE_MODE

    # Single config for "none" mode (no autotuning, fixed configuration)
    if mode == "none":
        return [
            triton.Config(
                {"BLOCK_SIZE": 64},
                num_stages=2,
                num_warps=4,
            )
        ]

    # Reduced configs for "medium" mode (fewer configurations for faster tuning)
    if mode == "medium":
        return [
            triton.Config(
                {"BLOCK_SIZE": BLOCK_SIZE},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE in [32, 64]  # Reduced set
            for num_stages in [2, 3]    # Reduced set
            for num_warps in [4, 8]     # Reduced set
        ]

    # Comprehensive configs for "max" mode (full search space)
    else:  # mode == "max" or any other value defaults to max
        return [
            triton.Config(
                {"BLOCK_SIZE": BLOCK_SIZE},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE in [16, 32, 64, 128]
            for num_stages in [2, 3, 4]
            for num_warps in [4, 8, 16]
        ]


In [ ]:
def get_bwd_autotune_configs():

    mode = AUTOTUNE_MODE

    # Single config for "none" mode (no autotuning, fixed configuration)
    if mode == "none":
        return [
            triton.Config(
                {"BLOCK_SIZE_MACRO": 64, "BLOCK_SIZE_MICRO": 32},
                num_stages=2,
                num_warps=4,
            )
        ]

    # Reduced configs for "medium" mode (fewer configurations for faster tuning)
    if mode == "medium":
        return [
            triton.Config(
                {"BLOCK_SIZE_MACRO": BLOCK_SIZE_MACRO, "BLOCK_SIZE_MICRO": BLOCK_SIZE_MICRO},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE_MACRO in [32, 64]  # Reduced set
            for BLOCK_SIZE_MICRO in [16, 32]  # Reduced set
            for num_stages in [2, 3]         # Reduced set
            for num_warps in [4, 8]          # Reduced set
            if BLOCK_SIZE_MICRO < BLOCK_SIZE_MACRO
        ]

    # Comprehensive configs for "max" mode (full search space)
    else:  # mode == "max" or any other value defaults to max
        return [
            triton.Config(
                {"BLOCK_SIZE_MACRO": BLOCK_SIZE_MACRO, "BLOCK_SIZE_MICRO": BLOCK_SIZE_MICRO},
                num_stages=num_stages,
                num_warps=num_warps,
            )
            for BLOCK_SIZE_MACRO in [16, 32, 64, 128]
            for BLOCK_SIZE_MICRO in [16, 32, 64]
            for num_stages in [2, 3, 4]
            for num_warps in [4, 8, 16]
            if BLOCK_SIZE_MICRO < BLOCK_SIZE_MACRO
        ]



In [ ]:
@triton.jit
def _attn_fwd_inner(
        O_block,
        l_i,
        m_i,
        Q_block,
        K_block_ptr,
        V_block_ptr,
        attn_mask,                      # masks are the shape (B x H x L x L)
        block_index_q,
        BLOCK_SIZE_Q,
        BLOCK_SIZE_KV,
        PASS_TYPE: tl.constexpr,        # 0: pre_diag, 1: diag, 2: full
        offs_q: tl.constexpr,
        offs_kv: tl.constexpr,
        SEQ_LEN,
        DTYPE_FLAG: tl.constexpr,
        USE_CUSTOM_MASK: tl.constexpr,
        BROADCAST_MASK_HEAD: tl.constexpr,
        stride_mask_batch,
        stride_mask_head,
        stride_mask_q,
        stride_mask_kv,
        index_batch,
        index_head,
):

    if PASS_TYPE == 0:
        lo, hi = 0, block_index_q * BLOCK_SIZE_Q

    elif PASS_TYPE == 1:
        lo, hi = block_index_q * BLOCK_SIZE_Q, (block_index_q+1) * BLOCK_SIZE_Q
        lo = tl.multiple_of(lo, BLOCK_SIZE_Q)

    else:   # passtype 2
        lo, hi = 0, SEQ_LEN


    K_block_ptr = tl.advance(K_block_ptr, (0, lo))
    V_block_ptr = tl.advance(V_block_ptr, (lo, 0))


    for start_kv in range(lo, hi, BLOCK_SIZE_KV):
        start_kv = tl.multiple_of(start_kv, BLOCK_SIZE_KV)

        kv_indices = start_kv + offs_kv
        kv_padding_mask = kv_indices < SEQ_LEN      # [block_size_kv]

        K_block = tl.load(K_block_ptr, boundary_check=(1,), padding_option="zero")
        V_block = tl.load(V_block_ptr, boundary_check=(0,), padding_option="zero")

        QK_block = tl.dot(Q_block, K_block)

        if PASS_TYPE == 1:      # diagonal

        # [0]    [0 1 2 3] -> [True False False False]
        # [1]                 [True True  False False]
        # [2]                 [True True  True  False]
        # [3]                 [True True  True  True]
        # and then we can just fill the False with a large negative number!

            causal_mask = offs_q[:, None] >= kv_indices[None, :]
            mask = causal_mask & kv_padding_mask[None, :]           # [1, block_size_kv]
            QK_block += tl.where(mask, 0, float("-inf"))

        else:
            QK_block += tl.where(kv_padding_mask[None, :], 0, float("-inf"))

        if USE_CUSTOM_MASK:

            # b x h x s x s, b x 1 x s x s
            mask_offset = (
                index_batch * stride_mask_batch +
                (0 if BROADCAST_MASK_HEAD else index_head) * stride_mask_head +
                offs_q[:, None] * stride_mask_q +
                kv_indices[None, :] * stride_mask_kv
            )

            custom_mask = tl.load(attn_mask_ptr + mask_offset,
                                  mask=(offs_q[:, None] < SEQ_LEN) & (kv_indices[None, :] < SEQ_LEN),
                                  other=False)

            QK_block += tl.where(custom_mask, 0, float("-inf"))

        m_ij = tl.maximum(m_i, tl.max(QK_block, 1))
        QK_block -= m_ij[:, None]

        P_block = tl.math.exp2(QK_block)
        l_ij = tl.sum(P_block, 1)
        alpha = tl.math.exp2(m_i - m_ij)
        l_i = l_i * alpha + l_ij

        P_block = P_block.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16)

        # use our formula to iteratively update our outputs O_new = PV + O_old * alpha
        O_block = O_block * alpha[:, None]
        O_block = tl.dot(P_block, V_block, acc=O_block)

        m_i = m_ij

        V_block_ptr = tl.advance(V_block_ptr, (BLOCK_SIZE_KV, 0))
        K_block_ptr = tl.advance(K_block_ptr, (0, BLOCK_SIZE_KV))

    return O_block, l_i, m_i




In [ ]:
@triton.autotune(
    configs = get_fwd_autotune_configs(),
    key=["HEAD_DIM"],
)

@triton.jit
def _attn_fwd(
        Q,
        K,
        V,
        attn_mask,
        softmax_scale: tl.constexpr,
        M,
        O,
        stride_Q_batch,
        stride_Q_head,
        stride_Q_seq,
        stride_Q_dim,
        stride_K_batch,
        stride_K_head,
        stride_K_seq,
        stride_K_dim,
        stride_V_batch,
        stride_V_head,
        stride_V_seq,
        stride_V_dim,
        stride_O_seq,
        stride_O_dim,
        stride_mask_batch,
        stride_mask_head,
        stride_mask_q,
        stride_mask_kv,
        NUM_HEADS: tl.constexpr,
        NUM_KV_HEADS: tl.constexpr,
        SEQ_LEN,
        HEAD_DIM: tl.constexpr,
        BLOCK_SIZE_Q: tl.constexpr,
        BLOCK_SIZE_KV: tl.constexpr,
        ATTN_MODE: tl.constexpr,
        DTYPE_FLAG: tl.constexpr,
        USE_CUSTOM_MASK: tl.constexpr,
        BROADCAST_MASK_HEAD: tl.constexpr,
):

    tl.static_assert(BLOCK_SIZE_KV <= HEAD_DIM)
    INV_LN2: tl.constexpr = 1.442695040888      # approx 1 / ln(2)

    softmax_scale *= INV_LN2

    block_index_q = tl.program_id(0)            # what block of queries are we grabbing?
    index_batch_head = tl.program_id(1)

    index_batch = index_batch_head // NUM_HEADS
    index_head = index_batch_head % NUM_HEADS

    group_size = NUM_HEADS // NUM_KV_HEADS
    index_kv_head =  index_head // group_size

    # B x H x S x E
    q_offset = (
        index_batch.to(tl.int64) * stride_Q_batch + index_head.to(tl.int64) * stride_Q_head
    )
    kv_offset = (
        index_batch.to(tl.int64) * stride_K_batch + index_kv_head.to(tl.int64) * stride_K_head
    )


    Q_block_ptr = tl.make_block_ptr(
        base = Q + q_offset,
        shape = (SEQ_LEN, HEAD_DIM),
        strides = (stride_Q_seq, stride_Q_dim),
        offsets = (block_index_q * BLOCK_SIZE_Q, 0),
        block_shape = (BLOCK_SIZE_Q, HEAD_DIM),
        order = (1, 0)
    )

    V_block_ptr = tl.make_block_ptr(
        base = V + kv_offset,
        shape = (SEQ_LEN, HEAD_DIM),
        strides = (stride_V_seq, stride_V_dim),
        offsets = (0, 0)                            # when loading values we dont skip anything, as we will for loop over this in a bit
        block_shape = (BLOCK_SIZE_KV, HEAD_DIM),
        order = (1, 0)
    )

    K_block_ptr= tl.make_block_ptr(
        base = K + kv_offset,
        shape = (HEAD_DIM, SEQ_LEN),                # set shape to transpose dimension
        strides = (stride_K_dim, stride_K_seq),     # invert the stride
        offsets = (0, 0),
        block_shape = (HEAD_DIM, BLOCK_SIZE_KV),
        order = (0, 1)                              # we want contiguous memory along the HEAD_DIM first
    )

    O_block_ptr = tl.make_block_ptr(
        base = O + q_offset,
        shape = (SEQ_LEN, HEAD_DIM),
        strides = (stride_O_seq, stride_O_dim),
        offsets = (block_index_q * BLOCK_SIZE_Q, 0),
        block_shape = (BLOCK_SIZE_Q, HEAD_DIM),
        order = (1, 0),
    )


    offs_q = block_index_q * BLOCK_SIZE_Q + tl.arange(0, BLOCK_SIZE_Q)
    offs_kv = tl.arange(0, BLOCK_SIZE_KV)

    m_i = tl.full(shape=[BLOCK_SIZE_Q], value=float("-inf"), dtype=tl.float32)
    l_i = tl.full(shape=[BLOCK_SIZE_Q], value=0.0, dtype=tl.float32)

    O_block = tl.zeros([BLOCK_SIZE_Q, HEAD_DIM], dtype=tl.float32)

    Q_block = tl.load(O_block_ptr, boundary_check=(0,), padding_option="zero")
    Q_block *= softmax_scale
    Q_block = Q_block.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16)

    # passtype 0: prediagonal, 1 ondiagonal, 2 full
    pass_type = 0 if ATTN_MODE == 1 else 2

    O_block, l_i, m_i = _attn_fwd_inner(
        O_block,
        l_i,
        m_i,
        Q_block,
        K_block_ptr,
        V_block_ptr,
        attn_mask,
        block_index_q,
        BLOCK_SIZE_Q,
        BLOCK_SIZE_KV,
        pass_type,
        offs_q,
        offs_kv,
        SEQ_LEN,
        DTYPE_FLAG,
        USE_CUSTOM_MASK,
        BROADCAST_MASK_HEAD,
        stride_mask_batch,
        stride_mask_head,
        stride_mask_q,
        stride_mask_kv,
        index_batch,
        index_head
    )

    if ATTN_MODE == 1:
        O_block, l_i, m_i = _attn_fwd_inner(
            O_block,
            l_i,
            m_i,
            Q_block,
            K_block_ptr,
            V_block_ptr,
            attn_mask,
            block_index_q,
            BLOCK_SIZE_Q,
            BLOCK_SIZE_KV,
            1,          # passtype = 1,
            offs_q,
            offs_kv,
            SEQ_LEN,
            DTYPE_FLAG,
            USE_CUSTOM_MASK,
            BROADCAST_MASK_HEAD,
            stride_mask_batch,
            stride_mask_head,
            stride_mask_q,
            stride_mask_kv,
            index_batch,
            index_head
        )


    m_i += tl.math.log2(l_i)
    O_block = O_block / (l_i[:, None] + 1e-6)

    # (B x H x S)
    m_ptrs = M + index_batch_head * SEQ_LEN + offs_q
    q_padding_mask = offs_q < SEQ_LEN
    tl.store(m_ptrs, m_i, mask=q_padding_mask)

    tl.store(O_block_ptr, O_block.to(O.type.element_ty), boundaty_check=(0,))





In [ ]:
def fused_sdpa_forward(Q, K, V,             # Q: (B, H, S, E), K,V: (B, kvH, S, E)
                       attn_mask=None,
                       causal=False,
                       softmax_scale=None):

    BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM_Q = Q.shape
    _, NUM_KV_HEADS, SEQ_LEN_KV, HEAD_DIM_K = K.shape
    _, _, _, HEAD_DIM_V = V.shape

    # sanity checks
    assert K.shape[1] == V.shape[1], "K and V must have the same number of heads!"

    assert NUM_HEADS % NUM_KV_HEADS == 0, (
        "number of query heads must be divisible by the number of key/value heads!"
    )

    assert HEAD_DIM_Q == HEAD_DIM_K = HEAD_DIM_V, (
        "Q, K, V head dimensions must match!"
    )

    assert SEQ_LEN_Q == SEQ_LEN_KV, "self attention expects Q, K, V to all have the same sequence length"

    assert Q.dtype == K.dtype == V.dtype, (
        "Q, K, V must have the same type!"
    )

    # (B x H x S x S), (B x 1 x S x S)
    if attn_mask is not None and attn_mask.shape[1] != 1:
        assert attn_mask.shape[1] == NUM_HEADS, "attention mask must either have 1 as head dim for broadcast, or match head dim in Queries"

    if softmax_scale is None:
        softmax_scale = 1 / HEAD_DIM_Q ** 0.5

    use_custom_mask = attn_mask is not None

    broadcast_mask_head = False
    if use_custom_mask:
        broadcast_mask_head = (attn_mask.shape[1] == 1)

    O = torch.empty_like(Q)

    # B x H x S x E
    grid = lambda args: (triton.cdiv(SEQ_LEN_Q, args["BLOCK_SIZE_Q"]), BATCH_SIZE*NUM_HEADS, 1)

    M = torch.empty(
        (BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q), dtype=torch.float32, device=Q.device
    )


    if use_custom_mask:
        stride_mask_batch, stride_mask_head, stride_mask_q, stride_mask_kv = attn_mask.stride()
    else:
        stride_mask_batch = 0
        stride_mask_head = 0
        stride_mask_q = 0
        stride_mask_kv = 0


    _attn_fwd[grid](
        Q=Q,
        K=K,
        V=V,
        attn_mask=attn_mask,
        softmax_scale=softmax_scale,
        M=M,
        O=O,
        stride_Q_batch=Q.stride(0),
        stride_Q_head=Q.stride(1),
        stride_Q_seq=Q.stride(2),
        stride_Q_dim=Q.stride(3),
        stride_K_batch=K.stride(0),
        stride_K_head=K.stride(1),
        stride_K_seq=K.stride(2),
        stride_K_dim=K.stride(3),
        stride_V_batch=V.stride(0),
        stride_V_head=V.stride(1),
        stride_V_seq=V.stride(2),
        stride_V_dim=V.stride(3),
        stride_O_seq=O.stride(2),
        stride_O_dim=O.stride(3),
        stride_mask_batch=stride_mask_batch,
        stride_mask_head=stride_mask_head,
        stride_mask_q=stride_mask_q,
        stride_mask_kv=stride_mask_kv,
        NUM_HEADS=NUM_HEADS,
        NUM_KV_HEADS=NUM_KV_HEADS,
        SEQ_LEN=SEQ_LEN,
        HEAD_DIM=HEAD_DIM,
        ATTN_MODE=1 if causal else 0,
        DTYPE_FLAG=0 if Q.dtype == torch.float32 else 1,
        USE_CUSTOM_MASK=use_custom_mask,
        BROADCAST_MASK_HEAD=broadcast_mask_head,

    )


    return Q, K, V, O, M



In [ ]:
@triton.jit
def _attn_bwd_dk_dv(
        K,
        V,
        dK,
        dV,
        Q_ptr,
        dO_ptr,
        M_ptr,
        D_ptr,
        attn_mask_ptr,
        stride_len,
        stride_embed,
        stride_mask_q,
        stride_mask_kv,
        SEQ_LEN,
        HEAD_DIM: tl.constexpr,
        BLOCK_SIZE_ROW: tl.constexpr,
        BLOCK_SIZE_COL: tl.constexpr,
        start_row,
        start_col,
        num_steps,
        ln2,
        MASK: tl.constexpr,
        DTYPE_FLAG: tl.constexpr,       # 0 for float32, 1 for float16
        USE_CUSTOM_MASK: tl.constexpr,
        mask_offset_base,
):
    """
    main method to compute the grads for dK, dV in blocks. this basically
    assumes for some K, V we loop through blocks of queries
    """

    offsets_row = start_row + tl.arange(0, BLOCK_SIZE_ROW)      # what queries do we want?
    offsets_col = start_col + tl.arange(0, BLOCK_SIZE_COL)      # what keys do we want?
    offsets_embed = tl.arange(0, HEAD_DIM)

    # B x H x S x E
    Q_T_offsets = offsets_embed[:, None] * stride_embed + offsets_row[None, :] * stride_len
    dO_offsets = offsets_row[:, None] * stride_len + offsets_embed[None, :] * stride_embed


    for _ in range(num_steps):

        mask_Q = offsets_row < SEQ_LEN

        # E x S
        Q_T_block = tl.load(Q_ptr + Q_T_offsets, mask=mask_Q[None, :], other=0.)

        M_block = tl.load(M_ptr + offsets_row, mask=mask_Q, other=0.)
        dO_block = tl.load(dO_ptr + dO_offsets, mask=mask_Q[:, None], other=0.)
        D_block = tl.load(D_ptr + offsets_row, mask=mask_Q, other=0.)

        S_T_block = tl.dot(K, Q_T_block)                            # (Macro x E) @ (E x Micro) -> Macro x Micro block

        P_T_block = tl.math.exp2(S_T_block - M_block[None, :])      # [block_kv, block_q]

        if MASK:    # causal mask
            # our P is transposed here. if causal, in the forward pass
            # only the lower triangle matters. this means in the backward
            # pass only the lower triangle matters, but because its transposed
            # we want the upper triangle instead!
            mask_block = (offsets_col[:, None] <= offsets_row[None, :])
            P_T_block = tl.where(mask_block, P_T_block, 0.0)

        # if we had an attention mask, we want to make sure we 0 out
        # any of the grads are coming from these masked positions!
        if USE_CUSTOM_MASK:

            # compute the indexes for the block of mask we want
            mask_offset = (
                mask_offset_base +
                offsets_row[None, :] * stride_mask_q +
                offsets_col[:, None] * stride_mask_kv
            )

            # grab that block of mask
            custom_mask = tl.load(attn_mask_ptr + mask_offset,
                                  mask=(offsets_row[None, :] < SEQ_LEN) & (offsets_col[:, None] < SEQ_LEN),
                                  other=False)

            # fill invalid positions with 0!
            P_T_block = tl.where(custom_mask, P_T_block, 0.)

        # now we start to accumulate grads. each block of the output contribute to our
        # gradient for dV. dV is P^T @ dO
        # but we are not processing all of our sequence length at once, only chunks of it
        # and our dV is dependent on contributions from the entire length so we can
        # just accumulate as we go for the correct positions we are processing
        dV = tl.dot(P_T_block.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16), dO_block, acc=dV)

        # dP = dO @ V^T, but we want dP^T so we transpose the right side and get [dO @ V^T]^T = V @ dO^T
        dP_T_block = tl.dot(V, tl.trans(dO_block))

        # then our dS = P*(dP - D) but we again have all transposes so we just use our transposed P and dP
        # D is just a row vector that is the broadcasted over, so we add an extra dimension to make it (1 x Micro)
        dS_T_block = P_T_block * (dP_T_block - D_block[None, :]) * ln2
        dK = tl.dot(dS_T_block.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16), tl.trans(Q_T_block), acc=dK)

        # advance to the next query block
        offsets_row += BLOCK_SIZE_ROW
        Q_ptr += BLOCK_SIZE_ROW * stride_len
        dO_ptr += BLOCK_SIZE_ROW * stride_len

    return dK, dV



In [ ]:
@triton.jit
def _attn_bwd_dq(
        dQ,
        Q,
        dO,
        M,
        K_ptr,
        V_ptr,
        D_ptr,
        attn_mask_ptr,

        stride_len,
        stride_embed,
        stride_mask_q,
        stride_mask_kv,

        SEQ_LEN,
        HEAD_DIM: tl.constexpr,
        BLOCK_SIZE_ROW: tl.constexpr,
        BLOCK_SIZE_COL: tl.constexpr,
        start_row,
        start_col,
        num_steps,
        ln2: tl.constexpr,
        MASK: tl.constexpr,
        DTYPE_FLAG: tl.constexpr,       # 0 for float32, 1 for float16
        USE_CUSTOM_MASK: tl.constexpr,
        mask_offset_base,
):
    """
    nearly identical for _attn_bwd_dk_dv but now we have a block of Q and are
    looping through blocks of K, V to compute out dQ. and istead of computing
    some transpose of our blocks of the attention matrix, we compute the normal
    non-transposed version as thats all we need
    """
    offsets_row = start_row + tl.arange(0, BLOCK_SIZE_ROW)
    offsets_col = start_col + tl.arange(0, BLOCK_SIZE_COL)
    offsets_embed = tl.arange(0, HEAD_DIM)

    K_V_T_offsets = offsets_embed[:, None] * stride_embed + offsets_col[None, :] * stride_len
    D_block = tl.load(D_ptr + offsets_row, mask=offsets_row<SEQ_LEN, other=0.)

    for _ in range(num_steps):

        # dont grab invalid masks
        mask_kv = offsets_col < SEQ_LEN

        K_T_block = tl.load(K_ptr + K_V_T_offsets, mask=mask_kv[None, :], other=0.)
        V_T_block = tl.load(V_ptr + K_V_T_offsets, mask=mask_kv[None, :], other=0.)

        # compute our standard QK^T
        S = tl.dot(Q, K_T_block)

        # logsumexp trick to get our softmax values back
        P = tl.exp2(S - M)

        # mask for causality
        if MASK:
            mask = offsets_row[:, None] >= offsets_col[None, :]
            P = tl.where(mask, P, 0.)

        # custom attention mask
        if USE_CUSTOM_MASK:

            mask_offset = (
                mask_offset_base +
                offsets_row[:, None] * stride_mask_q +
                offsets_col[None, :] * stride_mask_kv
            )

            custom_mask = tl.load(attn_mask_ptr + mask_offset,
                                  mask=(offsets_row[:, None] < SEQ_LEN) & (offsets_col[None, :] < SEQ_LEN),
                                  other=False)

            P = tl.where(custom_mask, P, 0.)

        # same formulation just for dQ now
        dP = tl.dot(dO, V_T_block)
        dS = P * (dP - D_block[:, None]) * ln2
        dQ = tl.dot(dS.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16), tl.trans(K_T_block), acc=dQ)

        # advance to the next block of keys/values
        offsets_col += BLOCK_SIZE_COL
        K_ptr += BLOCK_SIZE_COL * stride_len
        V_ptr += BLOCK_SIZE_COL * stride_len

    return dQ



In [ ]:
@triton.autotune(
    configs=get_bwd_autotune_configs(),
    key=["HEAD_DIM"],
)

@triton.jit
def _attn_bwd(
        Q_ptr,
        K_ptr,
        V_ptr,
        dO_ptr,
        dQ_ptr,
        dK_ptr,
        dV_ptr,
        M_ptr,
        D_ptr,
        attn_mask_ptr,
        softmax_scale,

        stride_batch,
        stride_head,
        stride_len,
        stride_embed,

        stride_k_batch,
        stride_k_head,
        stride_k_len,
        stride_k_embed,

        stride_mask_batch,
        stride_mask_head,
        stride_mask_q,
        stride_mask_kv,

        NUM_HEADS,
        SEQ_LEN,
        HEAD_DIM: tl.constexpr,
        BLOCK_SIZE_MICRO: tl.constexpr,
        BLOCK_SIZE_MACRO: tl.constexpr,
        CAUSAL: tl.constexpr,           # 1 for causal, 0 for noncausal
        DTYPE_FLAG: tl.constexpr,       # 0 for float32, 1 for float16
        USE_CUSTOM_MASK: tl.constexpr,
        BROADCAST_MASK_HEAD: tl.constexpr,
):

    tl.static_assert(BLOCK_SIZE_MACRO % BLOCK_SIZE_MICRO == 0)

    ln2: tl.constexpr = 0.693147182464
    rln2: tl.constexpr = 1.442695040888

    pid = tl.program_id(0)

    index_batch_head = tl.program_id(1)
    offsets_embed = tl.arange(0, HEAD_DIM)
    idx_batch = index_batch_head // NUM_HEADS
    idx_head = index_batch_head % NUM_HEADS
    group_size = NUM_HEADS // NUM_KV_HEADS
    idx_kv_head = idx_head // group_size

    # Q: (B x H x S x E)
    # KV: (B x kvH x S x E)
    # MD: (B x H x S)

    offset_batch_head_4d = idx_batch * stride_batch + idx_head * stride_head
    offset_batch_head_4d_kv = idx_batch * stride_k_batch + idx_kv_head * stride_k_head
    offset_batch_head_3d = index_batch_head * SEQ_LEN

    Q_ptr += offset_batch_head_4d
    K_ptr += offset_batch_head_4d_kv
    V_ptr += offset_batch_head_4d_kv
    dO_ptr += offset_batch_head_4d
    dQ_ptr += offset_batch_head_4d
    dK_ptr += offset_batch_head_4d_kv
    dV_ptr += offset_batch_head_4d_kv
    M_ptr += offset_batch_head_3d
    D_ptr += offset_batch_head_3d


    mask_offset_base = 0
    if USE_CUSTOM_MASK:
        mask_offset_base = (
            idx_batch * stride_mask_batch +
            (0 if BROADCAST_MASK_HEAD else idx_head) * stride_mask_head
        )


    # dK, dV
    # rows are the number of queries in every block we loop over
    # cols are the number of keys/values in our block that we hold constant in this specific thread
    BLOCK_SIZE_ROW_1: tl.constexpr = BLOCK_SIZE_MICRO       # block_size_q  128
    BLOCK_SIZE_COL_1: tl.constexpr = BLOCK_SIZE_MACRO       # block_size_kv 256

    if CAUSAL == 1:     # first do the diagonal
        # index of the starting key value index
        start_col = pid * BLOCK_SIZE_COL_1
        start_row = start_col
        num_steps = BLOCK_SIZE_COL_1 // BLOCK_SIZE_ROW_1

    else:
        start_row = 0
        start_col = pid * BLOCK_SIZE_COL_1
        num_steps = tl.cdiv(SEQ_LEN, BLOCK_SIZE_ROW_1)

    # what kv are we interested in?
    offsets_col_1 = start_col + tl.arange(0, BLOCK_SIZE_COL_1)

    KV_offsets = offsets_col_1[:, None] * stride_k_len + offsets_embed[None, :] * stride_k_embed
    KV_mask = (offsets_col_1 < SEQ_LEN)

    K = tl.load(K_ptr + KV_offsets, mask=KV_mask[:, None], other=0.)
    V = tl.load(V_ptr + KV_offsets, mask=KV_mask[:, None], other=0.)

    K *= softmax_scale * rln2
    K = K.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16)

    dK_block = tl.zeros([BLOCK_SIZE_COL_1, HEAD_DIM], dtype=tl.float32)
    dV_block = tl.zeros([BLOCK_SIZE_COL_1, HEAD_DIM], dtype=tl.float32)


    dK_block, dV_block = _attn_bwd_dk_dv(
        K, V, dK_block, dV_block,
        Q_ptr, dO_ptr, M_ptr, D_ptr, attn_mask_ptr,
        stride_len, stride_embed,
        stride_mask_q, stride_mask_kv,
        SEQ_LEN, HEAD_DIM,
        BLOCK_SIZE_ROW_1, BLOCK_SIZE_COL_1,
        start_row, start_col, num_steps,
        ln2,
        MASK=(CAUSAL==1),
        DTYPE_FLAG=DTYPE_FLAG,
        USE_CUSTOM_MASK=USE_CUSTOM_MASK,
        mask_offset_base=mask_offset_base,
    )


    ### STAGE 2: Process Under the Diagonal Block for Causal###
    ### If we are in causal mode we need to do all the other off diagonal positions. ###
    ### Now lets say we had the following block setup. Remember, each block has more ###
    ### values inside it but we are processing at the block level

    ### [B_00, B_01, B_02, B_03]
    ### [B_10, B_11, B_12, B_13]
    ### [B_20, B_21, B_22, B_23]
    ### [B_30, B_31, B_32, B_33]

    ### and lets say we processed B_11 just now (a diagonal block with transition from causal to non causal positions)
    ### Each thread here processes a column, as we picked a specific Key/Value, and we can loop through our queries.
    ### This means if we are causal, we need to also process B_21, and B_31! So lets move our starting row forward
    ### however many values there were in our keys/values and keep iterating downwards.


    if CAUSAL == 1:

        # if we have 256 kv blocks and 64 q blocks
        # lets say start_row = 256
        # kv: 256 -> 512
        # 256 + 64 -> 256 + 128
        # 256 + 128 -> 256 + 192
        # 256 + 196 -> 256 + 256

        start_row += BLOCK_SIZE_COL_1

        N_adj = tl.cdiv(SEQ_LEN, BLOCK_SIZE_COL_1) * BLOCK_SIZE_COL_1

        num_steps = (N_adj - start_row) // BLOCK_SIZE_ROW_1

        # backward pass again on these blocks underneath that diagonal block for the causal case
        dK_block, dV_block = _attn_bwd_dk_dv(
            K, V, dK_block, dV_block,
            Q_ptr, dO_ptr, M_ptr, D_ptr, attn_mask_ptr,
            stride_len, stride_embed,
            stride_mask_q, stride_mask_kv,
            SEQ_LEN, HEAD_DIM,
            BLOCK_SIZE_ROW_1, BLOCK_SIZE_COL_1,
            start_row, start_col, num_steps,
            ln2,
            MASK=False,
            DTYPE_FLAG=DTYPE_FLAG,
            USE_CUSTOM_MASK=USE_CUSTOM_MASK,
            mask_offset_base=mask_offset_base,
        )

    dK_block *= softmax_scale * rln2

    if NUM_HEADS == NUM_KV_HEADS:
        tl.store(dK_ptr + KV_offsets, dK_block, mask=KV_mask[:, None])
        tl.store(dV_ptr + KV_offsets, dV_block, mask=KV_mask[:, None])
    else:
        tl.atomic_add(dK_ptr + KV_offsets, dK_block, mask=KV_mask[:, None])
        tl.atomic_add(dV_ptr + KV_offsets, dV_block, mask=KV_mask[:, None])


    #################### dQ ####################
    # now we are grabbing some Q and looping over K, Vs. so we will
    # have Macro blocks of Q and loop through micro blocks of KVs
    BLOCK_SIZE_ROW_2: tl.constexpr = BLOCK_SIZE_MACRO
    BLOCK_SIZE_COL_2: tl.constexpr = BLOCK_SIZE_MICRO

    # stage 1: process the diagonal block
    # same setup as before, just now we go the other direction
    # our pid sets which block of rows of queries we grab
    # and our diagonal K/V will have the same starting point along the cols
    if CAUSAL == 1:
        start_row = pid * BLOCK_SIZE_ROW_2
        start_col = start_row
        num_steps = BLOCK_SIZE_ROW_2 // BLOCK_SIZE_COL_2

    # in non causal we just process everything
    else:
        start_col = 0
        start_row = pid * BLOCK_SIZE_ROW_2
        num_steps = tl.cdiv(SEQ_LEN, BLOCK_SIZE_COL_2)

    # compute offsets to grab our queries (also applies to our outputs)
    offsets_row = start_row + tl.arange(0, BLOCK_SIZE_ROW_2)
    Q_offsets = offsets_row[:, None] * stride_len + offsets_embed[None, :] * stride_embed

    # mask out any invalid queries we grabbed
    mask_row = offsets_row < SEQ_LEN

    # load our queries
    Q_block = tl.load(Q_ptr + Q_offsets, mask=mask_row[:, None], other=0.)

    # prescale our queries
    Q_block *= softmax_scale * rln2
    Q_block = Q_block.to(tl.float32 if DTYPE_FLAG == 0 else tl.float16)

    # laod our gradients for this specific block
    dO_block = tl.load(dO_ptr + Q_offsets, mask=mask_row[:, None], other=0.)

    M_block = tl.load(M_ptr + offsets_row, mask=mask_row, other=0.)[:, None]

    dQ_block = tl.zeros([BLOCK_SIZE_ROW_2, HEAD_DIM], dtype=tl.float32)

    dQ_block = _attn_bwd_dq(
        dQ_block, Q_block, dO_block, M_block,
        K_ptr, V_ptr, D_ptr, attn_mask_ptr,
        stride_k_len, stride_k_embed,
        stride_mask_q, stride_mask_kv,
        SEQ_LEN, HEAD_DIM,
        BLOCK_SIZE_ROW_2, BLOCK_SIZE_COL_2,
        start_row, start_col, num_steps,
        ln2, MASK=(CAUSAL==1),
        DTYPE_FLAG=DTYPE_FLAG,
        USE_CUSTOM_MASK=USE_CUSTOM_MASK,
        mask_offset_base=mask_offset_base,
    )

    # second pass (only for causal models)
    if CAUSAL == 1:
        end_col = start_col
        start_col = 0
        num_steps = end_col // BLOCK_SIZE_COL_2
        dQ_block = _attn_bwd_dq(
            dQ_block, Q_block, dO_block, M_block,
            K_ptr, V_ptr, D_ptr, attn_mask_ptr,
            stride_k_len, stride_k_embed,
            stride_mask_q, stride_mask_kv,
            SEQ_LEN, HEAD_DIM,
            BLOCK_SIZE_ROW_2, BLOCK_SIZE_COL_2,
            start_row, start_col, num_steps,
            ln2, MASK=False,
            DTYPE_FLAG=DTYPE_FLAG,
            USE_CUSTOM_MASK=USE_CUSTOM_MASK,
            mask_offset_base=mask_offset_base,
        )

    # scale our grads with the same factor
    dQ_block *= softmax_scale * rln2

    tl.store(dQ_ptr + Q_offsets, dQ_block, mask=mask_row[:, None])

In [ ]:
@triton.autotune(
    config=get_preprocess_autotune_configs(),
    key=["HEAD_DIM"],
)

@triton.jit
def attn_backward_preprocess(
    O_ptr,
    dO_ptr,
    D_ptr,
    stride_O_heads,
    stride_O_len,
    stride_O_embed,
    stride_dO_heads,
    stride_dO_len,
    stride_dO_embed,
    stride_D_head,
    SEQ_LEN,
    EMBED_DIM: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,       # 0 for float32, 1 for float16
):
    """
    just a fancy way to do sum(dO * O, axis=-1)
    """
    # (B x H x S x E)
    row = tl.program_id(0)
    index_batch_head = tl.program_id(1)

    row_offsets = row * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    col_offsets = tl.arange(0, EMBED_DIM)
    mask = (row_offsets < SEQ_LEN)

    # grab our output values
    O_ptr += index_batch_head * stride_O_heads
    O_offsets = row_offsets[:, None] * stride_O_len + col_offsets[None, :] * stride_O_embed
    O = tl.load(O_ptr + O_offsets, mask=mask[:, None], other=0.)

    # grab out output grads
    dO_ptr += index_batch_head * stride_dO_heads
    dO_offsets = row_offsets[:, None] * stride_dO_len + col_offsets[None, :] * stride_dO_embed
    dO = tl.load(dO_ptr + dO_offsets, mask=mask[:, None], other=0.)

    Delta = tl.sum(dO.to(tl.float32) * O.to(tl.float32), axis=1)
    D_ptr += index_batch_head * stride_D_head
    tl.store(D_ptr + row_offsets, Delta, mask=mask)








In [ ]:
def fused_sdpa_backward(
        dO,
        Q, K, V,
        O, M,
        attn_mask=None,
        causal=False,
        softmax_scale=None
):
    BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM_Q = Q.shape
    _, NUM_KV_HEADS, SEQ_LEN_KV, HEAD_DIM_K = K.shape
    _, _, _, HEAD_DIM_V = V.shape

    # sanity check
    assert K.shape[1] == V.shape[1], "K and V must have the same number of heads!"
    assert NUM_HEADS % NUM_KV_HEADS == 0, (
        "number of query heads must be divisible by the number of key/value heads!"
    )
    assert HEAD_DIM_Q == HEAD_DIM_K == HEAD_DIM_V, (
        "Q, K, V head dimensions must match!"
    )
    assert SEQ_LEN_Q == SEQ_LEN_KV, "self attention expects Q, K, V to all have the same sequence length"
    assert Q.dtype == K.dtype == V.dtype, (
        "Q, K, V must have the same data type!"
    )

    if attn_mask is not None and attn_mask.shape[1] != 1:
        assert attn_mask.shape[1] == NUM_HEADS, "attention mask must either have 1 as head dim for broadcast, or match head dim in Queries"

    # default softmax scale if not provided
    if softmax_scale is None:
        softmax_scale = 1 / HEAD_DIM_Q ** 0.5


    # check if we have attention mask
    use_custom_mask = attn_mask is not None

    broadcast_mask_head = False
    if use_custom_mask:
        broadcast_mask_head = (attn_mask.shape[1] == 1)

    # ensure grads have the same dtype
    if not dO.dtype == Q.dtype:
        dO = dO.astype(Q.dtype)

    # create empty grads to populate
    dQ = torch.zeros_like(Q, dtype=Q.dtype)
    dK = torch.zeros_like(K, dtype=K.dtype)
    dV = torch.zeros_like(V, dtype=V.dtype)

    # preprocess grid = (SEQ_LEN // BLOCK_SIZE_MACRO, BATCH_SIZE * NUM_HEADS)
    preprocess_grid = lambda meta: (triton.cdiv(SEQ_LEN_Q, meta["BLOCK_SIZE"]), BATCH_SIZE * NUM_HEADS)

    # this will contain our sum(O * dO, axis=-1) intermediate computation
    # do we need a kernel for this? probably not, but thats fine!
    D = torch.empty_like(M, dtype=torch.float32)        # shape: (BATCH_SIZE, NUM_HEADS, SEQ_LEN)


    attn_backward_preprocess[preprocess_grid](
        O_ptr=O,
        dO_ptr=dO,
        D_ptr=D,
        stride_O_heads=O.stride(1),
        stride_Q_len=Q.stride(2),
        stride_O_embed=O.stride(3),
        stride_dO_heads=dO.stride(1),
        stride_dO_len=dO.stride(2),
        stride_dO_embed=dO.stride(3),
        stride_D_head=D.stride(1),
        SEQ_LEN=SEQ_LEN_Q,
        EMBED_DIM=HEAD_DIM_Q,
        DTYPE_FLAG=0 if dO.dtype == torch.float32 else 1,
    )

    # mask strides
    if use_custom_mask:
        stride_mask_batch = attn_mask.stride(0)
        stride_mask_head = attn_mask.stride(1)
        stride_mask_q = attn_mask.stride(2)
        stride_mask_kv = attn_mask.stride(3)
        attn_mask_ptr = attn_mask.data_ptr()

    else:
        stride_mask_batch = 0
        stride_mask_head = 0
        stride_mask_q = 0
        stride_mask_kv = 0
        attn_mask_ptr = None

    grid = lambda meta: (
        triton.cdiv(SEQ_LEN_Q, meta["BLOCK_SIZE_MACRO"]),
        BATCH_SIZE * NUM_HEADS,
    )

    _attn_bwd[grid](
        Q_ptr=Q,
        K_ptr=K,
        V_ptr=V,
        dO_ptr=dO,
        dQ_ptr=dQ,
        dK_ptr=dK,
        dV_ptr=dV,
        M_ptr=M,
        D_ptr=D,
        attn_mask_ptr=attn_mask_ptr,
        softmax_scale=softmax_scale,

        stride_batch=Q.stride(0),
        stride_head=Q.stride(1),
        stride_len=Q.stride(2),
        stride_embed=Q.stride(3),

        stride_k_batch=K.stride(0),
        stride_k_head=K.stride(1),
        stride_k_len=K.stride(2),
        stride_k_embed=K.stride(3),

        stride_mask_batch=stride_mask_batch,
        stride_mask_head=stride_mask_head,
        stride_mask_q=stride_mask_q,
        stride_mask_kv=stride_mask_kv,

        NUM_HEADS=NUM_HEADS,
        NUM_KV_HEADS=NUM_KV_HEADS,
        SEQ_LEN=SEQ_LEN_Q,
        HEAD_DIM=HEAD_DIM_Q,
        CAUSAL=1 if causal else 0,
        DTYPE_FLAG=0 if Q.dtype == torch.float32 else 1,
        USE_CUSTOM_MASK=use_custom_mask,
        BROADCAST_MASK_HEAD=broadcast_mask_head
    )

    return dQ, dK, dV
